# Rung 1 — recomputing every claim from the run's own artifacts

This notebook *proves*; `summary.ipynb` *explains*. Nothing here is imported from this project:
every number below is recomputed from a written table using the Python standard library and
pandas, with the arithmetic written out in full, so what you read is exactly what is computed. A
notebook that only called the verification script would relocate the trust rather than discharge
it; that script appears once, in the last cell, as a cross-check.

**What this rung measures.** For one dose (5 micromolar) and a complete grid of 107 drugs by 50
cell lines, each round hides one line — or one drug — and every model predicts the change in
expression for the pairs it never saw. A prediction is scored by its correlation across genes
with the measured change, per pair, averaged over pairs, on two gene sets (the pair's responding
genes, and all of them). Models are compared head to head as the mean paired difference in that
score, with intervals, p-values and minimum detectable effects from 2,000 redraws of the held-out
units.

**What cannot be checked here, stated rather than hidden.** The 2,000 redraws per contrast live
in the run's scratch cache as `redraws_{b}.parquet`, not in the task folder: set `RUNG1_CACHE` to
that cache and the intervals, p-values and MDEs below are recomputed from them; without it those
cells say so and move on. The answers, cell descriptions and Stack embeddings are cluster-side
and far too large to commit, so what is checked of them is the parameter sidecar's sha256 of each
input that is present here. A grid smaller than 50 x 107 is a test fixture, and the claims that
are only true of the design's grid say so rather than passing quietly.

**Before the run.** The artifacts are uncommitted between the run and promotion (PROCESS, "What
reaches GitHub, and when"). Every cell degrades to "artifact not present yet" until they arrive,
so this notebook executes end to end on a fresh checkout. Set `RUNG1_TASK_DIR` to point it at
another run's tables and `RUNG1_CACHE` at that run's cache.


In [ ]:
import csv
import hashlib
import json
import math
import os
import statistics
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo(start: Path) -> Path:
    """The repository root: the first ancestor carrying pyproject.toml."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return start


REPO = find_repo(Path.cwd().resolve())
TASK = "rung1-held-out-prediction"
TASK_DIR = Path(os.environ.get("RUNG1_TASK_DIR") or (REPO / "docs" / "tasks" / TASK))
CACHE_DIR = Path(os.environ["RUNG1_CACHE"]) if os.environ.get("RUNG1_CACHE") else None
RUNG0_PER_PAIR = REPO / "results" / "rung0-assay-reliability" / "rung0_per_pair_r.csv"

#: design.md section 2: one dose, and a complete grid.
DOSE_UM = 5.0
DESIGN_LINES, DESIGN_DRUGS = 50, 107

#: design.md section 6: the ceiling a prediction is read against, and the pairs behind it.
DESIGN_CEILING = {"responding": 0.8575, "all": 0.3876}

#: design.md section 7: the statistics, and the two pairs removed for every model.
N_DRAWS, N_BLOCKS, MDE_FACTOR = 2000, 8, 2.8
REDRAW_BASE_SEED = 7000
SCIPLEX_LINE = "ACH-000681"
SCIPLEX_DRUGS = {"Temsirolimus", "Trametinib"}

#: design.md section 5: the candidate counts the two tuned descriptions -- and their stand-ins --
#: choose between, and the neighbour counts nearest lines chooses between.
COMPONENT_KS = [2, 5, 10, 15, 20]
NEAREST_LINES_KS = [3, 5, 10, 20]

#: design.md section 7's table: 6 leave-one-line-out comparisons, 5 leave-one-drug-out.
DECLARED = {
    "lolo": [
        "lolo_stack_base_vs_drug_average",
        "lolo_stack_base_vs_expression",
        "lolo_stack_base_vs_pca",
        "lolo_stack_base_vs_nmf",
        "lolo_stack_base_vs_nearest_lines",
        "lolo_stack_drug_vs_stack_cytokine",
    ],
    "lodo": [
        "lodo_stack_base_vs_chemistry_only",
        "lodo_stack_base_vs_expression",
        "lodo_stack_base_vs_pca",
        "lodo_stack_base_vs_nmf",
        "lodo_stack_drug_vs_stack_cytokine",
    ],
}

#: design.md section 8: one figure per step, each drawn from tables the run wrote.
FIGURES = {
    "01_build.png": ["rung1_cells.csv", "rung1_identity_match.csv", "rung1_weights_check.json"],
    "02_split.png": ["rung1_pair_scores.csv.gz", "rung1_control_split.csv"],
    "03_fit.png": ["rung1_settings.csv", "rung1_model_summary.csv", "rung1_control_fit.csv"],
    "04_score.png": ["rung1_model_summary.csv", "rung1_ceiling.csv", "rung1_control_score.csv"],
    "05_null.png": ["rung1_comparisons.csv"],
}

GENE_SETS = ["responding", "all"]
SCHEMES = ["lolo", "lodo"]

#: design.md section 7 also reports, unadjusted, each ridge description against its own random
#: stand-in -- the control behind H1(b). Enumerated here the way the run builds the ids, so a
#: run that reported none of them cannot pass a claim whose text says it covers them.
STAND_IN_DESCRIPTIONS = ["expression", "pca", "nmf", "stack_base", "stack_cytokine", "stack_drug"]
STAND_INS = [f"{s}_{d}_vs_random_{d}" for s in SCHEMES for d in STAND_IN_DESCRIPTIONS]

print(f"repository  {REPO}")
print(f"task dir    {TASK_DIR}")
print(f"cache       {CACHE_DIR}")
print(f"present     {(TASK_DIR / 'rung1_model_summary.csv').exists()}")

### The helpers, written out once

A handful of small functions and nothing else: read a table (plain or gzipped) into a data frame
or a list of dictionaries, decide whether a reported value is the recomputed one, and print
claim / recomputed / verdict. `read_table` returns `None` when the run has not happened yet,
which is what lets every cell below degrade to a message instead of a traceback.

Two details that matter for correctness. One of the screen's fifty cell lines has a missing
DepMap identifier and appears throughout as the literal string `NA`, so the default missing-value
list is off and only an empty cell counts as missing. And pandas' default CSV float parser is not
correctly rounded — it can move the last bit of a double — so every table is read with
`float_precision="round_trip"`; without it a mean recomputed here differs from the run's own in
the 16th digit for no reason but the reader.


In [ ]:
#: A recomputation agrees with the run's own arithmetic to within floating-point association;
#: this is not a rounding allowance but the width of that difference.
TOLERANCE = 1e-9


def read_table(name, directory=None):
    """A written table as a data frame, or None when it has not been written yet."""
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"artifact not present yet: {path}")
        return None
    return pd.read_csv(path, keep_default_na=False, na_values=[""], float_precision="round_trip")


def read_json(name, directory=None):
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"artifact not present yet: {path}")
        return None
    return json.loads(path.read_text())


def close(claim, recomputed, tolerance=TOLERANCE):
    """True when the reported value is the recomputed one, to within `tolerance`."""
    if math.isnan(claim) or math.isnan(recomputed):
        return math.isnan(claim) and math.isnan(recomputed)
    return abs(claim - recomputed) <= tolerance


def verdict(name, claim, recomputed, ok):
    print(f"{name}\n  claim      : {claim}\n  recomputed : {recomputed}")
    print(f"  verdict    : {'PASS' if ok else 'FAIL'}\n")


def sha256_of(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def sha256_items(items):
    """The grid record's hash of a name list: sha256 of the sorted names, newline-joined."""
    return hashlib.sha256("\n".join(sorted(items)).encode("utf-8")).hexdigest()


def find_grid():
    """The grid record, from the task folder or the run's cache.

    Some runs mirror it into the cache and read it from there, so both are legitimate places for
    it to be; only its absence from both is worth saying out loud.
    """
    for directory in (TASK_DIR, CACHE_DIR):
        if directory is None:
            continue
        path = directory / "rung1_grid.json"
        if path.exists():
            return json.loads(path.read_text())
    print(f"artifact not present yet: {TASK_DIR / 'rung1_grid.json'}")
    return None


grid = find_grid()
ceiling = read_table("rung1_ceiling.csv")
pair_scores = read_table("rung1_pair_scores.csv.gz")
model_summary = read_table("rung1_model_summary.csv")
comparisons = read_table("rung1_comparisons.csv")
settings = read_table("rung1_settings.csv")
params = read_json("rung1_run.params.json")

## Claim 1 — the grid is one dose, a complete line-by-drug rectangle, and hashes to what it says

`rung1_grid.json` is the restriction record: which lines and drugs were scored, which pairs were
removed, and the sha256 of each. The counts are checked against design section 2's 107 drugs by
50 lines, and the two hashes are recomputed from the name lists they pin — a record whose hash no
longer matches its own names is a record that was edited after the run.


In [ ]:
if grid:
    dose = float(grid.get("dose", float("nan")))
    verdict(
        "the grid is at one dose, 5 uM",
        f"rung1_grid.json records dose={grid.get('dose')}",
        f"design.md section 2 fixes the dose at {DOSE_UM} uM",
        close(dose, DOSE_UM),
    )

    lines, drugs = list(grid["lines"]), list(grid["drugs"])
    design_size = not (len(lines) < DESIGN_LINES and len(drugs) < DESIGN_DRUGS)
    if design_size:
        verdict(
            "the grid is 107 drugs x 50 lines = 5,350 pairs",
            f"{len(drugs)} drugs and {len(lines)} lines in the record",
            f"{len(lines)} x {len(drugs)} = {len(lines) * len(drugs):,} pairs",
            (len(lines), len(drugs)) == (DESIGN_LINES, DESIGN_DRUGS),
        )
    else:
        print(
            f"this run's grid is {len(lines)} lines x {len(drugs)} drugs, a fixture rather than "
            f"the design's {DESIGN_LINES} x {DESIGN_DRUGS}; the size claim does not apply\n"
        )

    for key, items in (("sha256_lines", lines), ("sha256_drugs", drugs)):
        recorded = grid.get(key, "")
        if not recorded:
            print(f"rung1_grid.json records no {key}; nothing to recompute\n")
            continue
        verdict(
            f"{key} recomputes from the names it pins",
            f"{key} {recorded}",
            f"sha256 of the {len(items)} sorted names: {sha256_items([str(i) for i in items])}",
            recorded == sha256_items([str(i) for i in items]),
        )

## Claim 2 — the ceiling is rung 0's reliability, restricted to this grid, and square-rooted

A prediction is scored against one noisy measurement, so the best it can reach is not rung 0's
split-half agreement R but its square root: two measurements meet the noise twice, a prediction
once. The run's `rung1_ceiling.csv` is built from rung 0's promoted per-pair table, limited to
this grid's pairs at 5 uM: average the per-pair split-half correlation, lift it to full data with
Spearman-Brown `2r / (1 + r)`, take the square root. Both steps are redone here from the
committed table — the one number rung 1 inherits rather than measures.


In [ ]:
if ceiling is not None:
    for _, row in ceiling.iterrows():
        split_half = float(row["split_half_r"])
        sb, sqrt_sb = float(row["sb"]), float(row["sqrt_sb"])
        expected_sb = 2 * split_half / (1 + split_half)
        verdict(
            f"{row['gene_set']}: sb is 2r/(1+r) of the split-half mean, sqrt_sb its square root",
            f"sb {sb}, sqrt_sb {sqrt_sb}",
            f"2 x {split_half:.4f} / (1 + {split_half:.4f}) = {expected_sb:.4f}, "
            f"sqrt {math.sqrt(expected_sb):.4f}",
            close(sb, expected_sb, 1e-4) and close(sqrt_sb, math.sqrt(sb), 1e-4),
        )
    declared = {str(r["gene_set"]): float(r["sqrt_sb"]) for _, r in ceiling.iterrows()}
    verdict(
        "the ceilings are design section 6's declared values",
        str(declared),
        str(DESIGN_CEILING),
        all(close(declared.get(k, float("nan")), v, 5e-5) for k, v in DESIGN_CEILING.items()),
    )

if ceiling is not None and grid and RUNG0_PER_PAIR.exists():
    lines, drugs = set(map(str, grid["lines"])), set(map(str, grid["drugs"]))
    values = {"responding": [], "all": []}
    with RUNG0_PER_PAIR.open(newline="") as handle:
        for row in csv.DictReader(handle):
            if row["patient"] not in lines or row["drug"] not in drugs:
                continue
            if float(row["dose"]) != DOSE_UM:
                continue
            for gene_set, column in (("responding", "r_responder"), ("all", "r")):
                text = (row[column] or "").strip()
                if text:
                    values[gene_set].append(float(text))
    # The grid itself is rung 0's complete 5 uM sub-table: every line it measured at that dose,
    # and every drug measured in all of them. Re-derived here rather than taken on trust.
    at_dose = {}
    with RUNG0_PER_PAIR.open(newline="") as handle:
        for row in csv.DictReader(handle):
            if float(row["dose"]) == DOSE_UM:
                at_dose.setdefault(row["drug"], set()).add(row["patient"])
    all_lines = sorted({line for covered in at_dose.values() for line in covered})
    complete = sorted(d for d, covered in at_dose.items() if len(covered) == len(all_lines))
    if set(grid["lines"]) & set(all_lines):
        verdict(
            "the grid's lines and drugs are rung 0's complete 5 uM grid",
            f"{len(grid['lines'])} lines and {len(grid['drugs'])} drugs in rung1_grid.json",
            f"rung 0 gives {len(all_lines)} lines and {len(complete)} drugs measured in all of "
            "them",
            list(map(str, grid["lines"])) == all_lines
            and list(map(str, grid["drugs"])) == complete,
        )

    if not values["all"]:
        print(
            "none of this run's pairs is in rung 0's 5 uM table, so this grid is a fixture "
            "rather than the screen's; the ceiling cannot be rebuilt from it here\n"
        )
    else:
        for gene_set in GENE_SETS:
            scored = values[gene_set]
            mean_r = statistics.fmean(scored)
            sb = 2 * mean_r / (1 + mean_r)
            row = ceiling.loc[ceiling["gene_set"] == gene_set].iloc[0]
            verdict(
                f"{gene_set}: the ceiling rebuilds from rung 0's promoted per-pair table",
                f"{int(row['pairs_scored_rung0']):,} pairs, r {float(row['split_half_r']):.4f}, "
                f"√SB {float(row['sqrt_sb']):.4f}",
                f"{len(scored):,} pairs, r {mean_r:.4f}, √SB {math.sqrt(sb):.4f}",
                int(row["pairs_scored_rung0"]) == len(scored)
                and close(float(row["split_half_r"]), mean_r, 5e-5)
                and close(float(row["sqrt_sb"]), math.sqrt(sb), 5e-5),
            )

## Claim 3 — every model's mean score is the mean over the pairs *every* model scored

A comparison only means something if the models are compared on the same pairs, so a pair any
model left unscored (fewer than 50 usable genes, or no variance on one side) is dropped for all
of them. That population is rebuilt here from the written per-pair scores — one model per column,
drop any row with a gap — and each model's reported mean, pair count and fraction of the ceiling
is recomputed over it. The summary is keyed by scheme as well as model and gene set: the same
model fitted with a line hidden and with a drug hidden is answering two different questions.


In [ ]:
population = {}
if pair_scores is not None:
    for scheme in SCHEMES:
        for gene_set in GENE_SETS:
            chosen = (pair_scores["scheme"] == scheme) & (pair_scores["gene_set"] == gene_set)
            rows = pair_scores.loc[chosen]
            if rows.empty:
                continue
            wide = rows.pivot(index=["line", "drug"], columns="model", values="r")
            population[(scheme, gene_set)] = wide.dropna(axis=0, how="any")
    for (scheme, gene_set), wide in sorted(population.items()):
        models = len(wide.columns)
        print(f"{scheme}/{gene_set}: {len(wide):,} pairs all {models} models scored")
    print()

if model_summary is not None and population:
    reported = {
        (str(r["scheme"]), str(r["model"]), str(r["gene_set"])) for _, r in model_summary.iterrows()
    }
    expected = {(s, str(m), g) for (s, g), wide in population.items() for m in wide.columns}
    verdict(
        "one summary row per scheme, model and gene set",
        f"{len(model_summary)} rows in rung1_model_summary.csv",
        f"{len(expected)} (scheme, model, gene set) combinations in the pair scores",
        reported == expected and len(model_summary) == len(expected),
    )

    for (scheme, gene_set), wide in sorted(population.items()):
        rows = model_summary.loc[
            (model_summary["scheme"] == scheme) & (model_summary["gene_set"] == gene_set)
        ].set_index("model")
        worst_mean = worst_fraction = 0.0
        outside = []
        # A model with no summary row is a disagreement to report, not an exception to die of:
        # a reviewer's notebook that raises here tells them nothing about the other models.
        absent = [str(m) for m in wide.columns if str(m) not in rows.index]
        for model in wide.columns:
            if str(model) in absent:
                continue
            mean_r = float(wide[model].mean())
            row = rows.loc[str(model)]
            worst_mean = max(worst_mean, abs(float(row["mean_r"]) - mean_r))
            worst_fraction = max(
                worst_fraction,
                abs(float(row["fraction_of_ceiling"]) - mean_r / float(row["sqrt_sb"])),
            )
            if not float(row["ci_lo"]) <= mean_r <= float(row["ci_hi"]):
                outside.append(str(model))
        verdict(
            f"{scheme}/{gene_set}: every mean score is the mean of its per-pair scores",
            f"{len(rows)} models over {len(wide):,} shared pairs",
            f"largest difference {worst_mean:.3e}"
            + (f"; models with no summary row: {absent}" if absent else ""),
            worst_mean <= TOLERANCE and not absent,
        )
        verdict(
            f"{scheme}/{gene_set}: every fraction of the ceiling is that mean over √SB",
            f"{len(rows)} fractions",
            f"largest difference {worst_fraction:.3e}; means outside their interval: {outside}",
            worst_fraction <= TOLERANCE and not outside,
        )

## Claim 4 — every comparison is the mean paired difference between two models' scores

Each row of `rung1_comparisons.csv` says one model beat another by so much. That is the average,
over the shared pairs, of one model's per-pair score minus the other's — recomputed here from the
same population. The design's eleven declared comparisons must all be present, on both gene sets,
beside the description-versus-random-stand-in contrasts it reports unadjusted.


In [ ]:
if comparisons is not None and population:
    present = set(map(str, comparisons["comparison"]))
    missing = [c for family in DECLARED.values() for c in family if c not in present]
    missing_stand_ins = [c for c in STAND_INS if c not in present]
    per_contrast = comparisons.groupby("comparison").size()
    declared_count = sum(len(f) for f in DECLARED.values())
    verdict(
        "every declared comparison AND every stand-in contrast is reported, on both gene sets",
        f"{len(present)} contrasts, {len(comparisons)} rows",
        f"design section 7 declares {declared_count} comparisons and {len(STAND_INS)} "
        f"description-versus-stand-in contrasts, {declared_count + len(STAND_INS)} in all; "
        f"missing comparisons {missing}; missing stand-in contrasts {missing_stand_ins}; "
        f"rows per contrast {sorted(set(per_contrast.tolist()))}",
        not missing and not missing_stand_ins and set(per_contrast.tolist()) == {len(GENE_SETS)},
    )

    for gene_set in GENE_SETS:
        rows = comparisons.loc[comparisons["gene_set"] == gene_set]
        worst, trouble = 0.0, []
        for _, row in rows.iterrows():
            wide = population.get((str(row["scheme"]), gene_set))
            if wide is None or str(row["model_a"]) not in wide or str(row["model_b"]) not in wide:
                trouble.append(str(row["comparison"]))
                continue
            difference = wide[str(row["model_a"])] - wide[str(row["model_b"])]
            worst = max(worst, abs(float(row["estimate"]) - float(difference.mean())))
            if int(row["n_pairs"]) != len(difference):
                trouble.append(f"{row['comparison']} (n_pairs)")
        verdict(
            f"{gene_set}: every estimate is the mean paired difference",
            f"{len(rows)} estimates",
            f"largest difference {worst:.3e}" + (f"; trouble {trouble}" if trouble else ""),
            worst <= TOLERANCE and not trouble,
        )

## Claim 5 — the intervals, p-values and minimum detectable effects are the redraws' own

Uncertainty here comes from redrawing the held-out units with repeats — the 50 lines when a line
is hidden, the 107 drugs when a drug is hidden — 2,000 times, and recomputing each comparison.
The interval is the middle 95% of those redraws, the p-value twice the smaller share of them on
one side of zero (each share counted with one added draw, so it is never exactly zero), and the
minimum detectable effect 2.8 of their standard deviations: the smallest true difference a
two-sided 5% test would find 80% of the time.

The draws live in the run's cache, one parquet per block of 250. Set `RUNG1_CACHE` and this cell
recomputes all four quantities; without it, it says so.


In [ ]:
redraws = None
if CACHE_DIR is None:
    print("RUNG1_CACHE is not set, so the redraw blocks were not read; claim 5 is not checked\n")
else:
    blocks = [CACHE_DIR / f"redraws_{b}.parquet" for b in range(N_BLOCKS)]
    absent = [b.name for b in blocks if not b.exists()]
    if absent:
        print(f"{len(absent)} of {N_BLOCKS} redraw blocks are absent from {CACHE_DIR}")
        print(f"{absent[:3]}; claim 5 is not checked\n")
    else:
        redraws = pd.concat([pd.read_parquet(b) for b in blocks], ignore_index=True)
        print(f"read {len(redraws):,} redraws from {N_BLOCKS} blocks in {CACHE_DIR}\n")

if redraws is not None and comparisons is not None:
    counts = redraws.groupby(["comparison", "gene_set"])["draw"].agg(["size", "nunique", "max"])
    complete = (
        (counts["size"] == N_DRAWS)
        & (counts["nunique"] == N_DRAWS)
        & (counts["max"] == N_DRAWS - 1)
    )
    verdict(
        "every contrast holds all 2,000 draws, numbered once each across the blocks",
        f"{len(counts)} (contrast, gene set) pairs",
        f"{int(complete.sum())} of {len(counts)} hold draws 0..{N_DRAWS - 1} exactly once",
        bool(complete.all()) and len(counts) > 0,
    )

    worst = {"interval": 0.0, "p": 0.0, "mde": 0.0, "design effect": 0.0}
    for _, row in comparisons.iterrows():
        chosen = (redraws["comparison"] == row["comparison"]) & (
            redraws["gene_set"] == row["gene_set"]
        )
        draws = redraws.loc[chosen].sort_values("draw")
        one_way = [v for v in draws["estimate"].tolist() if not math.isnan(v)]
        two_way = [v for v in draws["estimate_two_way"].tolist() if not math.isnan(v)]
        low, high = pd.Series(one_way).quantile([0.025, 0.975]).tolist()
        n = len(one_way)
        at_or_below = (1 + sum(1 for v in one_way if v <= 0)) / (1 + n)
        at_or_above = (1 + sum(1 for v in one_way if v >= 0)) / (1 + n)
        p = min(1.0, 2 * min(at_or_below, at_or_above))
        sd = statistics.stdev(one_way)
        effect = statistics.variance(two_way) / statistics.variance(one_way)
        worst["interval"] = max(
            worst["interval"], abs(float(row["ci_lo"]) - low), abs(float(row["ci_hi"]) - high)
        )
        worst["p"] = max(worst["p"], abs(float(row["p"]) - p))
        worst["mde"] = max(worst["mde"], abs(float(row["mde"]) - MDE_FACTOR * sd))
        worst["design effect"] = max(
            worst["design effect"],
            abs(float(row["design_effect"]) - effect),
            abs(float(row["width_ratio"]) - math.sqrt(effect)),
        )
    for what, difference in worst.items():
        verdict(
            f"every {what} recomputes from the redraws",
            f"{len(comparisons)} rows in rung1_comparisons.csv",
            f"largest difference {difference:.3e}",
            difference <= 1e-6,
        )

## Claim 6 — Holm's adjustment is applied within each test, and nowhere else

Six comparisons are read under the main test (a line hidden) and five under the second (a drug
hidden), so each family is adjusted for multiplicity on its own: sorted by p, the i-th smallest
of m is multiplied by m − i, kept monotone, and capped at 1. Design section 7 adjusts the
responding-gene rows of the declared comparisons only — the all-genes rows and every stand-in
contrast are reported unadjusted, and their `p_holm` is left empty rather than filled with the
raw value.


In [ ]:
def holm(p_values):
    """Holm's step-down adjustment, in the input order."""
    m = len(p_values)
    order = sorted(range(m), key=lambda i: p_values[i])
    adjusted, running = [0.0] * m, 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted


if comparisons is not None:
    adjusted = comparisons.loc[comparisons["p_holm"].notna()]
    for scheme in SCHEMES:
        family = adjusted.loc[adjusted["scheme"] == scheme].sort_values("comparison")
        if family.empty:
            print(f"no adjusted {scheme} rows in rung1_comparisons.csv\n")
            continue
        recomputed = holm([float(v) for v in family["p"]])
        reported = [float(v) for v in family["p_holm"]]
        worst = max(abs(a - b) for a, b in zip(recomputed, reported, strict=True))
        verdict(
            f"{scheme}: Holm is recomputed within the test, over its own family",
            f"{len(family)} adjusted p-values; design section 7 declares {len(DECLARED[scheme])}",
            f"largest difference {worst:.3e}",
            worst <= TOLERANCE and len(family) == len(DECLARED[scheme]),
        )
    wrong = [
        str(row["comparison"])
        for _, row in adjusted.iterrows()
        if str(row["gene_set"]) != "responding"
        or str(row["comparison"]) not in DECLARED["lolo"] + DECLARED["lodo"]
    ]
    verdict(
        "nothing outside the declared responding-gene comparisons carries an adjusted p",
        f"{len(adjusted)} rows carry p_holm",
        "every one is a declared comparison on responding genes" if not wrong else f"also: {wrong}",
        not wrong,
    )

## Claim 7 — the pairs a model had already seen are scored for nobody

The sci-Plex fine-tune was trained on A549 (DepMap `ACH-000681`) with five compounds, two of
which are in this grid. Those pairs are removed for **every** model, not only for the fine-tune,
so that all models are still scored on the same pairs. This cell looks for them in the written
per-pair scores, where they must not appear at all.


In [ ]:
if grid and pair_scores is not None:
    excluded = [(str(a), str(b)) for a, b in grid["excluded_pairs"]]
    scored = set(zip(pair_scores["line"].astype(str), pair_scores["drug"].astype(str), strict=True))
    found = [pair for pair in excluded if pair in scored]
    if not excluded:
        print("this grid removed no pairs, so there is nothing to look for\n")
    else:
        verdict(
            "every removed pair is scored for no model, in either scheme",
            f"{len(excluded)} pairs removed: {excluded}",
            f"{len(found)} of them appear among {len(scored):,} scored pairs"
            + (f": {found}" if found else ""),
            not found,
        )
    if SCIPLEX_LINE in set(map(str, grid["lines"])):
        stripped = {(a, b.strip()) for a, b in excluded}
        verdict(
            "the removed pairs are the two design section 7 names",
            f"{sorted(stripped)}",
            f"{sorted((SCIPLEX_LINE, drug) for drug in SCIPLEX_DRUGS)}",
            stripped == {(SCIPLEX_LINE, drug) for drug in SCIPLEX_DRUGS},
        )
    else:
        print(f"{SCIPLEX_LINE} is not one of this run's lines, so those two pairs cannot be here\n")

leakage = read_json("rung1_leakage_profiles.json")
if leakage:
    versions = [str(p["model_version"]) for p in leakage]
    verdict(
        "one leakage record per Stack version, none of which saw treated cells",
        f"{len(leakage)} records: {versions}",
        "ruling 40: the three Stack versions; every other model is fitted in-run from training "
        "data alone",
        versions == ["stack_base", "stack_cytokine", "stack_drug"]
        and not any(bool(p["saw_tahoe_treated_cells"]) for p in leakage),
    )

## Claim 8 — every figure the design declared exists, and so do the tables it was drawn from

`design.md` section 8 names the figures before the run, so a reviewer sees the evidence the
design promised rather than the subset that looked best afterwards. Each must be a real PNG, and
each must have the tables it was drawn from beside it: a figure whose source table was never
written is a stage that did not run, and the two have to be told apart.


In [ ]:
figure_dir = TASK_DIR / "figures"
if figure_dir.exists():
    present = [name for name in FIGURES if (figure_dir / name).exists()]
    real = [
        name
        for name in present
        if (figure_dir / name).stat().st_size > 5_000
        and (figure_dir / name).read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"
    ]
    missing = sorted(set(FIGURES) - set(present))
    verdict(
        "every figure design section 8 declares was written, and is a real image",
        f"{len(FIGURES)} figures, each a PNG over 5 kB",
        f"{len(present)} present, {len(real)} non-trivial"
        + (f"; missing {missing}" if missing else ""),
        len(real) == len(FIGURES),
    )
    absent = {
        figure: [t for t in tables if not (TASK_DIR / t).exists()]
        for figure, tables in FIGURES.items()
    }
    absent = {figure: tables for figure, tables in absent.items() if tables}
    verdict(
        "every figure's source tables are in the task folder",
        f"{sum(len(t) for t in FIGURES.values())} source tables across {len(FIGURES)} figures",
        "all present" if not absent else f"absent: {absent}",
        not absent,
    )
else:
    print(f"artifact not present yet: {figure_dir}")

## Claim 9 — the provenance record: what was read, at which commit, with which seeds

The parameter sidecar is what ties a number to the bytes and the code that produced it: the
sha256 of every input the run read, the commit it ran at, every seed, and the candidate component
counts the tuned models actually chose from. The checksums are recomputed here for every input
present in this tree; the rest are compared with what the design and the rulings declare.


In [ ]:
#: What the sidecar has to carry for the claims below to mean anything. An absent key is as much
#: a gap as an absent file, and neither may stop this notebook halfway down with a KeyError.
REQUIRED_PARAMS = [
    "inputs",
    "seeds",
    "n_draws",
    "n_blocks",
    "draws_per_block",
    "component_ks",
    "git_sha",
]
if params:
    absent_keys = [key for key in REQUIRED_PARAMS if key not in params]
    if absent_keys:
        print(f"rung1_run.params.json records no {absent_keys}; every claim below that reads one")
        print("says so rather than being checked\n")

if params and "inputs" in params:
    roots = {"out_dir": TASK_DIR, "cache": CACHE_DIR}
    matched, moved, unresolved = [], [], []
    for key, digest in params["inputs"].items():
        root_name, _, relative = str(key).partition("/")
        root = roots.get(root_name)
        path = None if root is None else root / relative
        if path is None or not path.exists():
            unresolved.append(key)
        elif sha256_of(path) == digest:
            matched.append(key)
        else:
            moved.append(key)
    verdict(
        "every pinned input still hashes to what the run recorded",
        f"{len(params['inputs'])} input checksums in rung1_run.params.json",
        f"{len(matched)} recompute, {len(moved)} CHANGED ({moved[:3]}), {len(unresolved)} not in "
        "this tree (they live in the run's scratch cache)",
        not moved and bool(matched),
    )

if params and {"seeds", "n_draws", "n_blocks", "draws_per_block"} <= set(params):
    seeds = params["seeds"]
    recorded = {
        "redraw_base_seed": seeds.get("redraw_base_seed"),
        "n_draws": params["n_draws"],
        "n_blocks": params["n_blocks"],
        "draws_per_block": params["draws_per_block"],
    }
    declared = {
        "redraw_base_seed": REDRAW_BASE_SEED,
        "n_draws": N_DRAWS,
        "n_blocks": N_BLOCKS,
        "draws_per_block": N_DRAWS // N_BLOCKS,
    }
    verdict(
        "the redraws' base seed and block structure are the declared ones",
        str(recorded),
        f"ruling 33 and design section 7 declare {declared}",
        recorded == declared,
    )

if params and "component_ks" in params:
    component_ks = {str(m): [int(k) for k in ks] for m, ks in params["component_ks"].items()}
    design_size = bool(grid) and not (
        len(grid["lines"]) < DESIGN_LINES and len(grid["drugs"]) < DESIGN_DRUGS
    )
    if design_size:
        verdict(
            "every tuned model chose from the design's candidate component counts",
            str(component_ks),
            f"design section 5 declares {COMPONENT_KS} for pca, nmf and their stand-ins",
            all(ks == COMPONENT_KS for ks in component_ks.values()) and len(component_ks) == 4,
        )
    else:
        print(
            f"this run's grid is a fixture, where fewer candidates are legitimate: {component_ks}\n"
        )

if params and "git_sha" in params:
    sha = str(params["git_sha"])
    result = subprocess.run(
        ["git", "-C", str(REPO), "merge-base", "--is-ancestor", sha, "HEAD"],
        capture_output=True,
        check=False,
    )
    verdict(
        "the run was made at a commit on this branch",
        f"git_sha {sha[:12]}",
        f"an ancestor of HEAD: {result.returncode == 0}",
        result.returncode == 0,
    )

if params and ceiling is not None:
    recorded_ceiling = {str(k): float(v) for k, v in params.get("ceiling", {}).items()}
    from_table = {str(r["gene_set"]): float(r["sqrt_sb"]) for _, r in ceiling.iterrows()}
    verdict(
        "the ceiling the run recorded is the one rung1_ceiling.csv carries",
        str(recorded_ceiling),
        str(from_table),
        set(recorded_ceiling) == set(from_table)
        and all(close(recorded_ceiling[k], from_table[k]) for k in from_table),
    )

if settings is not None and grid:
    rounds = {"lolo": len(grid["lines"]), "lodo": len(grid["drugs"])}
    trouble = []
    for scheme, n_rounds in rounds.items():
        rows = settings.loc[settings["scheme"] == scheme]
        seen = sorted({int(v) for v in rows["round"]})
        if seen != list(range(n_rounds)):
            trouble.append(f"{scheme}: {len(seen)} rounds of {n_rounds}")
            continue
        per_round = rows.groupby("round")["model"].apply(lambda m: tuple(sorted(m)))
        if len(set(per_round)) != 1:
            trouble.append(f"{scheme}: the models fitted differ between rounds")
    verdict(
        "every round of both schemes recorded one settings row per model",
        f"{len(settings)} settings rows",
        f"{rounds['lolo']} leave-one-line-out and {rounds['lodo']} leave-one-drug-out rounds"
        + (f"; {trouble}" if trouble else ""),
        not trouble,
    )

if settings is not None and params and "component_ks" in params:
    chosen = settings.loc[settings["k"].notna()]
    candidates = {str(m): [int(k) for k in ks] for m, ks in params["component_ks"].items()}

    def candidates_for(model):
        """A component count comes from the run's own candidates; a neighbour count from the
        design's. The settings table's `k` column carries both, so they are never read against
        each other's set."""
        if model in candidates:
            return candidates[model]
        return NEAREST_LINES_KS if model == "nearest_lines" else []

    outside = sorted(
        {
            f"{row['model']} k={int(row['k'])}"
            for _, row in chosen.iterrows()
            if int(row["k"]) not in candidates_for(str(row["model"]))
        }
    )
    verdict(
        "every chosen setting comes from its own candidate set",
        f"{len(chosen)} rows of rung1_settings.csv chose a k",
        f"components {candidates}, nearest lines {NEAREST_LINES_KS}"
        + (f"; outside them: {outside}" if outside else ""),
        not outside,
    )

## Cross-check — the same claims through `scripts/verify_rung1.py`

Everything above was recomputed here, in the open. This last cell runs the project's verification
battery over the same artifacts, so the two paths are compared rather than one standing in for
the other. It is the notebook's cross-check, never its body: a notebook that only called this
script would relocate the trust instead of discharging it. The script exits 2 when the run has
not happened yet, which is what the message below reports on a fresh checkout.


In [ ]:
command = [sys.executable, str(REPO / "scripts" / "verify_rung1.py"), "--task-dir", str(TASK_DIR)]
if CACHE_DIR is not None:
    command += ["--cache", str(CACHE_DIR)]
completed = subprocess.run(command, capture_output=True, text=True, check=False)
print(completed.stdout or completed.stderr)
print(f"exit status {completed.returncode}")